# CSA — frozen text bank cache + real ROI smoke
Add Dataset `ducanh2403/csa-real-v126-lemma-fixed`; enable Internet and GPU (GPU0 only). No API secret.
Encodes 5,269 nouns / 1,594 verbs with pinned frozen CLIP. Then 2 XE smoke updates on 4 TRAIN images and beam generation on 2 val images.
This is not long training or quality evaluation. Randomly initialized captions are not research results.
Preserve the entire output folder, including `semantic_banks.npz`; send only the small review ZIP.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile
from pathlib import Path
PREPARED_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-real-v126-lemma-fixed')
OUTPUT=Path('/kaggle/working/csa_real_bank_cache_smoke')
COMMIT='086094e3d6a84cbe10b90ea237e290c7f38fd8d1'
HASHES={'real_bank_data.py': '00e7110105ec98a5ef41969f9e51074815a42a9b69deac90b2614e9679d2bcf9', 'run_real_bank.py': '30a15d904979513ef9351880c7763a1315ff817bdc7cb4043d034d5630d63fa5', 'REAL_BANK_INPUT_PACKAGE.json': 'c60902e04ea15cf90f652e384937bb317e288753f13b8ede6ab5ef89f8d97129', 'REAL_BANK_CACHE_PROTOCOL.md': 'd84e2f3f733f5c2b52c9fd1d57c1033019b098703f62389023bcd0e9939f06a8', 'model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09'}
if OUTPUT.exists(): raise RuntimeError('Preserve existing output; use a fresh session.')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='csa-real-bank-'))
LOG=OUTPUT/'notebook.log'
os.environ['HF_HOME']=str(SOURCE/'hf-cache')
os.environ['HF_HUB_DISABLE_XET']='1'
print('Pinned source:',COMMIT,flush=True)
for name,h in HASHES.items():
    payload=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{name}',timeout=90).read()
    if hashlib.sha256(payload).hexdigest()!=h: raise ValueError('Changed source: '+name)
    (SOURCE/name).write_bytes(payload)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT,sha256=HASHES),indent=2),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Worker active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    # No global dependency changes; preserve system Torch/NumPy.
    env=SOURCE/'text-env'
    run([sys.executable,'-m','venv','--without-pip','--system-site-packages',env],timeout=120)
    PY=env/'bin/python'
    run([sys.executable,'-m','pip','--python',PY,'install','--disable-pip-version-check',
         'transformers==4.57.1','huggingface-hub==0.36.2','tokenizers==0.22.1','ftfy==6.3.1'],timeout=900)
    shared=['--prepared-root',PREPARED_ROOT,'--output',OUTPUT,'--hf-cache',SOURCE/'hf-cache']
    run([PY,'-u',SOURCE/'run_real_bank.py','cache',*shared],timeout=1800)
    # Separate process frees CLIP weights before CSA allocation; successful cache persists.
    run([PY,'-u',SOURCE/'run_real_bank.py','smoke',*shared],timeout=600)
    complete=True
except Exception:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    raise
finally:
    review=Path(str(OUTPUT)+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as archive:
        for p in sorted(OUTPUT.iterdir()):
            if p.is_file() and p.suffix in ('.json','.txt','.log','.md'): archive.write(p,p.name)
    print('SEND REVIEW:',review,flush=True)
    print('KEEP ENTIRE CACHE FOLDER:',OUTPUT,flush=True)
    print('COMPLETE' if complete else 'INCOMPLETE; successful cache receipt remains valid if present.',flush=True)
